In [ ]:
# Guardar el modelo y la lista de features con comprobaciones
import pickle
import json

if 'rf_model' in globals():
metadata
cell_type
source
Para evitar ejecuciones recursivas que bloqueen el kernel, no ejecute jupyter nbconvert --execute desde dentro del mismo notebook.
Si desea ejecutar el notebook desde la línea de comandos, use PowerShell en la carpeta raíz del proyecto y corre:\n
````powershell\ncd "e:\\TPO - CIENCIA DE DATOS\\APP 2\
notebooks/data_analysis.ipynb" --output "notebooks/data_analysis_executed.ipynb"\n````
Esto ejecutará el notebook en un proceso externo y guardará la versión ejecutada como `data_analysis_executed.ipynb`.

⚠️ rf_model no está definido. Ejecuta las celdas de entrenamiento antes de guardar el modelo.
⚠️ features_to_use no está definido. Revisa la celda de preparación de datos.


# Data Analysis - Safe Notebook
Este notebook rehace el flujo de EDA y modelo en forma segura: no contiene llamadas a `nbconvert`, no ejecuta comandos del sistema que puedan bloquear el kernel y guarda los artefactos en `../backend/` y `../frontend/`.

Instrucciones: ejecutar celdas en orden. Las gráficas se guardan automáticamente; este notebook usa un backend no interactivo para matplotlib para evitar bloqueos en entornos sin display.

In [ ]:
# Imports y configuración segura
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend no interactivo
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('✓ Librerías importadas')

In [ ]:
# Rutas y comprobaciones básicas
ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'database.csv'
BACKEND = ROOT / 'backend'
FRONTEND = ROOT / 'frontend'
BACKEND.mkdir(parents=True, exist_ok=True)
FRONTEND.mkdir(parents=True, exist_ok=True)
print('DATA:', DATA_PATH)
assert DATA_PATH.exists(), 'database.csv no encontrado en la carpeta raíz del proyecto'

In [ ]:
# Cargar datos (carga segura y ligera, evitar imprimir todo)
df = pd.read_csv(DATA_PATH)
print('Dimensiones:', df.shape)
display(df.head(5))

In [ ]:
# Resumen y tipos
print(df.dtypes)
print('
Valores faltantes por columna:')
print(df.isnull().sum())

In [ ]:
# Gráfica de valores faltantes (si aplica)
missing = df.isnull().sum()
missing = missing[missing>0]
if len(missing):
    fig, ax = plt.subplots(figsize=(10,6))
    missing.sort_values().plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Count of missing values')
    fig.tight_layout()
    fig.savefig(FRONTEND / 'missing_values.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    print('✓ missing_values.png guardada')
else:
    print('✓ No missing values found')

In [ ]:
# Distribuciones simplificadas y guardado
fig, axes = plt.subplots(2,2, figsize=(12,9))
if 'seniority' in df.columns:
    df['seniority'].value_counts().plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title('Distribución de seniority')
if 'genero' in df.columns:
    df['genero'].value_counts().plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('Género')
if 'dedicacion' in df.columns:
    df['dedicacion'].value_counts().plot(kind='bar', ax=axes[1,0])
    axes[1,0].set_title('Dedicación')
if 'tengo_edad' in df.columns:
    df['tengo_edad'].hist(bins=30, ax=axes[1,1])
    axes[1,1].set_title('Edad')
fig.tight_layout()
fig.savefig(FRONTEND / 'distributions.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print('✓ distributions.png guardada')

## Preparación de datos para modelado
A continuación se seleccionan y limpian las features utilizadas por el modelo.

In [ ]:
# Selección de features (ajusta si tus nombres difieren)
features_to_use = [
    'tengo_edad',
    'Años de experiencia',
    'antiguedad_en_la_empresa_actual',
    'Años en el puesto actual',
    'cuantas_personas_tenes_a_cargo',
    'ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos',
    'ultimo_salario_mensual_o_retiro_neto_en_pesos_argentinos',
    'sueldo_bruto_en_dolares',
    'sueldo_neto_en_dolares'
]
# Filtrar solo las columnas existentes
features_to_use = [f for f in features_to_use if f in df.columns]
print('Features usadas:', features_to_use)
# Copia para modelado
df_model = df.copy()
# Rellenar NA con mediana en features numéricas
for c in features_to_use:
    if df_model[c].isnull().sum()>0:
        df_model[c].fillna(df_model[c].median(), inplace=True)
print('✓ NA manejados en features')

In [ ]:
# Preparar X e y y hacer split (80/20)
X = df_model[features_to_use]
y = df_model['seniority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train/test sizes:', X_train.shape[0], X_test.shape[0])

In [ ]:
# Entrenar RandomForest (rápido y con parámetros conservadores)
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=10, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('✓ Modelo entrenado')

In [ ]:
# Predicciones y métricas
y_pred = rf.predict(X_test)
train_acc = accuracy_score(y_train, rf.predict(X_train))
test_acc = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print('Train acc:', train_acc)
print('Test acc:', test_acc)
print(classification_report(y_test, y_pred))

In [ ]:
# Guardar gráficas de rendimiento y feature importance
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_title('Matriz de Confusión')
fig.tight_layout()
fig.savefig(FRONTEND / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.close(fig)
# Feature importance
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
fig, ax = plt.subplots(figsize=(8,6))
ax.barh(fi['feature'], fi['importance'], color='steelblue')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FRONTEND / 'feature_importance.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print('✓ Gráficas de confusión e importancia guardadas')

In [ ]:
# Guardar resultados y modelo (con comprobaciones)
analysis = {
    'model_metrics': {
        'train_accuracy': float(train_acc),
        'test_accuracy': float(test_acc),
        'precision': float(test_precision),
        'recall': float(test_recall),
        'f1_score': float(test_f1)
    },
    'dataset_info': {'
        'total_rows': int(df.shape[0]),
        'total_columns': int(df.shape[1]),
        'train_size': int(X_train.shape[0]),
        'test_size': int(X_test.shape[0])
    },
    'feature_importance': fi.to_dict('records')
}
(BACKEND / 'analysis_results.json').write_text(json.dumps(analysis, ensure_ascii=False, indent=2), encoding='utf-8')
with open(BACKEND / 'random_forest_model.pkl','wb') as f:
    pickle.dump(rf, f)
with open(BACKEND / 'features.json','w', encoding='utf-8') as f:
    json.dump({'features': list(X.columns)}, f, ensure_ascii=False, indent=2)
print('✓ analysis_results.json, random_forest_model.pkl y features.json guardados en backend')

## Fin - notas
Este notebook fue creado para ser seguro en entornos sin display y evitar llamadas peligrosas como `nbconvert`. Si quieres, puedo añadir pruebas unitarias o celdas de validación adicionales.